# xWAR — Test New Data (Hitters + Pitchers)

A single notebook for scoring **any** new hitter or pitcher dataset against the finalized linear xWAR
models, without touching the training notebooks (`xWAR_hitters.ipynb`, `xWAR_pitchers.ipynb`).

**How to use this notebook:**
1. Set `HITTER_TRAIN_FILE` / `PITCHER_TRAIN_FILE` (the historical training data — unchanged run to run)
   and `HITTER_TEST_FILE` / `PITCHER_TEST_FILE` (whatever new data you want to score) in the config
   cell below.
2. Run all cells. Each section trains its model fresh on the training file (identical feature set and
   cleaning logic to the finalized v5/v2 notebooks) and then scores the test file via `.predict()` only
   — the test data is never used to fit anything.
3. Leave either test file blank/unset (`None`) to skip that section if you only want to score hitters
   or only pitchers.

This mirrors the calibration logic in the two training notebooks exactly, so results here will match
what you'd get by re-running those notebooks on the same training data — the only thing that changes
is which file gets scored.

## Config — set your file names here

In [ ]:
# Historical training data (2021-2025) -- used to fit the models, unchanged run to run
HITTER_TRAIN_FILE = 'expectedwarhitters.csv'
PITCHER_TRAIN_FILE = 'expectedwarpitchers.csv'

# New data to score -- change these to whatever you want to test.
# Set to None to skip that section entirely.
HITTER_TEST_FILE = 'file.csv'
PITCHER_TEST_FILE = 'file.csv'

# Output file names
HITTER_OUTPUT_FILE = 'xwar_hitters_predictions.csv'
PITCHER_OUTPUT_FILE = 'xwar_pitchers_predictions.csv'

## Imports

In [ ]:
import pandas as pd
import numpy as np
from sklearn.linear_model import LinearRegression
from sklearn.metrics import r2_score, mean_absolute_error

pd.set_option('display.float_format', lambda x: f'{x:,.4f}')

---
# Part 1 — Hitters

Trains the finalized v5 hitting + baserunning linear models (see `xWAR_hitters.ipynb` for the full
feature-selection writeup) on `HITTER_TRAIN_FILE`, then scores `HITTER_TEST_FILE` as pure held-out
data. Handles multi-position rows the same way the training notebook does (collapsed into one row per
player, OAA summed across positions) — if your test file has a `player_id`/`season` column it collapses
on that; otherwise it falls back to `name` + `team`.

In [ ]:
if HITTER_TEST_FILE is not None:

    # --- Step 1: load + clean training data (identical to xWAR_hitters.ipynb v5) ---
    df = pd.read_csv(HITTER_TRAIN_FILE, na_values=['null'])

    pct_cols = ['bb', 'k', 'barrel', 'hardhit', 'ld', 'gb', 'fb', 'pull',
                'h_swing', 'c_swing', 's_swing', 'w_swing']
    for col in pct_cols:
        if col in df.columns:
            df[col] = df[col].astype(str).str.rstrip('%').astype(float) / 100
    df.rename(columns={'bb': 'bb_pct', 'k': 'k_pct', 'barrel': 'barrel_pct'}, inplace=True)

    dedup_pos = df.groupby(['player_id', 'season', 'position', 'team'], as_index=False).first()
    agg = dedup_pos.groupby(['player_id', 'season'], as_index=False).agg(
        name=('name', 'first'), pa=('pa', 'first'),
        bb_pct=('bb_pct', 'first'), k_pct=('k_pct', 'first'), barrel_pct=('barrel_pct', 'first'),
        sprint_speed=('sprint_speed', 'mean'), oaa=('oaa', 'sum'),
        woba=('woba', 'first'), bsr=('bsr', 'first'), war=('war', 'first'),
        pos=('position', 'first'), frm=('frm', 'max'),
        sb=('sb', 'first'), cs=('cs', 'first'),
        hardhit=('hardhit', 'first'), ev90=('ev90', 'first'),
        la=('la', 'first'), ld=('ld', 'first'), fb=('fb', 'first'),
    )
    train_h = agg.dropna(subset=['sprint_speed', 'barrel_pct', 'woba', 'bsr']).reset_index(drop=True)
    train_h['is_catcher'] = train_h['pos'] == 'C'
    league_woba = train_h['woba'].mean()   # single pooled league wOBA for scoring arbitrary test data

    print(f'[Hitters] Trained on {len(train_h)} player-seasons from {HITTER_TRAIN_FILE}')

    # --- Step 2: fit hitting model (xwOBA) ---
    hit_features = ['bb_pct', 'k_pct', 'barrel_pct', 'hardhit', 'ev90', 'la', 'ld', 'fb']
    lin_hit = LinearRegression().fit(train_h[hit_features], train_h['woba'])
    train_h['xwoba'] = lin_hit.predict(train_h[hit_features])

    # --- Step 3: fit baserunning model (xBsR) ---
    run_features = ['sprint_speed', 'xwoba', 'sb', 'cs']
    lin_run = LinearRegression().fit(train_h[run_features], train_h['bsr'])
    train_h['xbsr'] = lin_run.predict(train_h[run_features])

    # --- Step 4: fielding runs, positional adjustment, replacement level ---
    OAA_TO_RUNS = {'1B': 0.75, '2B': 0.75, '3B': 0.75, 'SS': 0.75, 'LF': 0.90, 'CF': 0.90, 'RF': 0.90}
    POSITIONAL_ADJ_PER_600 = {
        'C': 9.5, '1B': -9.5, '2B': 2.5, '3B': 2.0, 'SS': 7.5,
        'LF': -7.0, 'CF': 2.5, 'RF': -7.0, 'DH': -15.0
    }
    REPLACEMENT_RUNS_PER_600 = 20.0

    def add_component_runs(d):
        d['oaa_run_factor'] = d['pos'].map(OAA_TO_RUNS).fillna(0.80)
        d['oaa_runs'] = d['oaa'] * d['oaa_run_factor']
        d['fielding_runs'] = np.where(d['is_catcher'], d['frm'], d['oaa_runs']).astype(float)
        d['fielding_runs'] = d['fielding_runs'].fillna(0)
        d['pos_adj_runs'] = d['pos'].map(POSITIONAL_ADJ_PER_600).fillna(0) * (d['pa'] / 600)
        d['replacement_runs'] = REPLACEMENT_RUNS_PER_600 * (d['pa'] / 600)
        return d

    train_h = add_component_runs(train_h)

    # --- Step 5: calibrate WOBA_SCALE / RUNS_PER_WIN on TRAIN ONLY ---
    train_h['raw_woba_diff_term'] = (train_h['xwoba'] - league_woba) * train_h['pa']
    train_h['other_runs'] = train_h['xbsr'] + train_h['fielding_runs'] + train_h['pos_adj_runs'] + train_h['replacement_runs']

    calib_cols = ['raw_woba_diff_term', 'other_runs']
    calib_model = LinearRegression(fit_intercept=False).fit(train_h[calib_cols], train_h['war'])
    c1, c2 = calib_model.coef_
    WOBA_SCALE = c2 / c1
    RUNS_PER_WIN = 1 / c2

    print(f'[Hitters] WOBA_SCALE = {WOBA_SCALE:.4f}   RUNS_PER_WIN = {RUNS_PER_WIN:.4f}')
    print(f'[Hitters] Train calib R^2 = {calib_model.score(train_h[calib_cols], train_h["war"]):.4f}')
else:
    print('[Hitters] HITTER_TEST_FILE is None -- skipping hitter section.')

In [ ]:
if HITTER_TEST_FILE is not None:

    # --- Step 6: load + clean TEST data (pure inference -- never touches .fit()) ---
    raw = pd.read_csv(HITTER_TEST_FILE, na_values=['null'])

    pct_cols_test = ['bb', 'k', 'barrel', 'hardhit', 'ld', 'gb', 'fb', 'pull']
    for col in pct_cols_test:
        if col in raw.columns:
            raw[col] = raw[col].astype(str).str.rstrip('%').astype(float) / 100
    raw.rename(columns={'bb': 'bb_pct', 'k': 'k_pct', 'barrel': 'barrel_pct',
                         'sb%': 'sb_pct', 'pos': 'pos'}, inplace=True)

    # Collapse multi-position rows into one row per player, same idea as the training aggregation.
    # Groups on player_id+season if available, otherwise falls back to name+team.
    group_keys = ['player_id', 'season'] if {'player_id', 'season'}.issubset(raw.columns) else ['name', 'team']

    test_h = raw.groupby(group_keys, as_index=False).agg(
        name=('name', 'first'), team=('team', 'first'), pa=('pa', 'first'),
        bb_pct=('bb_pct', 'first'), k_pct=('k_pct', 'first'), barrel_pct=('barrel_pct', 'first'),
        sprint_speed=('sprint_speed', 'mean'), oaa=('oaa', 'sum'),
        war=('war', 'first') if 'war' in raw.columns else ('pa', 'first'),
        pos=('pos', 'first'), frm=('frm', 'max'),
        sb=('sb', 'first'), cs=('cs', 'first'),
        hardhit=('hardhit', 'first'), ev90=('ev90', 'first'),
        la=('la', 'first'), ld=('ld', 'first'), fb=('fb', 'first'),
    )
    if 'war' not in raw.columns:
        test_h = test_h.drop(columns=['war'])

    test_h['is_catcher'] = test_h['pos'] == 'C'
    test_h = test_h.dropna(subset=['sprint_speed', 'barrel_pct']).reset_index(drop=True)
    print(f'[Hitters] Scoring {len(test_h)} player-seasons from {HITTER_TEST_FILE} (held out)')

    # --- Step 7: score (predict only) ---
    test_h['xwoba'] = lin_hit.predict(test_h[hit_features])
    test_h['xbsr'] = lin_run.predict(test_h[run_features])
    test_h = add_component_runs(test_h)

    test_h['raw_woba_diff_term'] = (test_h['xwoba'] - league_woba) * test_h['pa']
    test_h['other_runs'] = test_h['xbsr'] + test_h['fielding_runs'] + test_h['pos_adj_runs'] + test_h['replacement_runs']

    def compute_xwar_hitter(d):
        batting_runs = d['raw_woba_diff_term'] / WOBA_SCALE
        rar = batting_runs + d['other_runs']
        return rar, rar / RUNS_PER_WIN

    test_h['RAR'], test_h['xWAR'] = compute_xwar_hitter(test_h)

    out_cols = ['name', 'team', 'pos', 'pa', 'xwoba', 'xbsr', 'fielding_runs', 'xWAR']
    if 'war' in test_h.columns:
        out_cols.append('war')
    result_h = test_h[out_cols].sort_values('xWAR', ascending=False).reset_index(drop=True)
    result_h.to_csv(HITTER_OUTPUT_FILE, index=False)

    print(f'[Hitters] Saved predictions to {HITTER_OUTPUT_FILE}')
    print(result_h.head(20).to_string(index=False))

    if 'war' in test_h.columns:
        print(f"\n[Hitters] xWAR vs actual WAR -- R^2 = {r2_score(test_h['war'], test_h['xWAR']):.4f}   MAE = {mean_absolute_error(test_h['war'], test_h['xWAR']):.4f}")

---
# Part 2 — Pitchers

Trains the finalized v2 xRA9 linear model (see `xWAR_pitchers.ipynb` for the feature-selection
writeup — no features were dropped in that review) on `PITCHER_TRAIN_FILE`, then scores
`PITCHER_TEST_FILE` as pure held-out data.

In [ ]:
if PITCHER_TEST_FILE is not None:

    # --- Step 1: load + clean training data (identical to xWAR_pitchers.ipynb v2) ---
    dfp = pd.read_csv(PITCHER_TRAIN_FILE)

    pct_cols_p = ['GB%', 'HR/FB', 'FB%', 'K%', 'BB%', 'HardHit%']
    for col in pct_cols_p:
        if col in dfp.columns:
            dfp[col] = dfp[col].astype(str).str.rstrip('%').astype(float) / 100

    dfp = dfp.rename(columns={
        'K%': 'k_pct', 'BB%': 'bb_pct', 'HR/FB': 'hr_fb_pct',
        'GB%': 'gb_pct', 'FB%': 'fb_pct', 'HardHit%': 'hardhit_pct',
        'IP': 'ip', 'ERA': 'era', 'WAR': 'war',
    })

    ra9_features = ['k_pct', 'bb_pct', 'hr_fb_pct', 'gb_pct', 'fb_pct', 'hardhit_pct']
    train_p = dfp.dropna(subset=ra9_features + ['era']).reset_index(drop=True)
    print(f'[Pitchers] Trained on {len(train_p)} player-seasons from {PITCHER_TRAIN_FILE}')

    # --- Step 2: fit xRA9 model ---
    lin_ra9 = LinearRegression().fit(train_p[ra9_features], train_p['era'])
    train_p['xra9'] = lin_ra9.predict(train_p[ra9_features])

    # --- Step 3: calibrate RA9_replacement / RUNS_PER_WIN on TRAIN ONLY ---
    train_p['ip_per9'] = train_p['ip'] / 9
    train_p['xra'] = train_p['xra9'] * train_p['ip_per9']

    calib_model_p = LinearRegression(fit_intercept=False).fit(train_p[['ip_per9', 'xra']], train_p['war'])
    a, b = calib_model_p.coef_
    RUNS_PER_WIN_P = 1 / -b
    RA9_REPLACEMENT = a * RUNS_PER_WIN_P

    print(f'[Pitchers] RA9_replacement = {RA9_REPLACEMENT:.4f}   RUNS_PER_WIN = {RUNS_PER_WIN_P:.4f}')
    print(f'[Pitchers] Train calib R^2 = {calib_model_p.score(train_p[["ip_per9","xra"]], train_p["war"]):.4f}')
else:
    print('[Pitchers] PITCHER_TEST_FILE is None -- skipping pitcher section.')

In [ ]:
if PITCHER_TEST_FILE is not None:

    # --- Step 4: load + clean TEST data (pure inference -- never touches .fit()) ---
    test_p = pd.read_csv(PITCHER_TEST_FILE)

    pct_cols_test_p = ['K%', 'BB%', 'HR/FB', 'GB%', 'FB%', 'HardHit%']
    for col in pct_cols_test_p:
        if col in test_p.columns:
            test_p[col] = test_p[col].astype(str).str.rstrip('%').astype(float) / 100

    test_p = test_p.rename(columns={
        'K%': 'k_pct', 'BB%': 'bb_pct', 'HR/FB': 'hr_fb_pct',
        'GB%': 'gb_pct', 'FB%': 'fb_pct', 'HardHit%': 'hardhit_pct',
        'IP': 'ip', 'ERA': 'era', 'WAR': 'war',
    })
    test_p = test_p.dropna(subset=ra9_features + ['ip']).reset_index(drop=True)
    print(f'[Pitchers] Scoring {len(test_p)} pitchers from {PITCHER_TEST_FILE} (held out)')

    # --- Step 5: score (predict only) ---
    test_p['xra9'] = lin_ra9.predict(test_p[ra9_features])
    test_p['ip_per9'] = test_p['ip'] / 9

    def compute_xwar_pitcher(d):
        rar = (RA9_REPLACEMENT - d['xra9']) * (d['ip'] / 9)
        return rar, rar / RUNS_PER_WIN_P

    test_p['RAR'], test_p['xWAR'] = compute_xwar_pitcher(test_p)

    name_col = 'Name' if 'Name' in test_p.columns else 'name'
    team_col = 'Team' if 'Team' in test_p.columns else 'team'
    out_cols = [name_col, team_col, 'ip', 'xra9', 'xWAR']
    if 'war' in test_p.columns:
        out_cols.append('war')
    result_p = test_p[out_cols].sort_values('xWAR', ascending=False).reset_index(drop=True)
    result_p.to_csv(PITCHER_OUTPUT_FILE, index=False)

    print(f'[Pitchers] Saved predictions to {PITCHER_OUTPUT_FILE}')
    print(result_p.head(20).to_string(index=False))

    if 'war' in test_p.columns:
        print(f"\n[Pitchers] xWAR vs actual WAR -- R^2 = {r2_score(test_p['war'], test_p['xWAR']):.4f}   MAE = {mean_absolute_error(test_p['war'], test_p['xWAR']):.4f}")

## Notes

- Both sections use plain relative filenames — put `HITTER_TRAIN_FILE`, `PITCHER_TRAIN_FILE`, and
  whatever you set `HITTER_TEST_FILE`/`PITCHER_TEST_FILE` to in the same folder as this notebook.
- Test data never touches `.fit()` anywhere in this notebook — every model call on test data is
  `.predict()` only, consistent with the finalized training notebooks.
- If your test file is missing `war`/`WAR`, the output simply skips the actual-vs-predicted comparison
  and R²/MAE summary — you'll still get the xWAR leaderboard.
- The hitter section's `league_woba` is a single pooled average across the training data rather than a
  season-by-season lookup, since test data may not have (or need) a `season` column. If your test data
  does have known season-specific league context you want to use instead, swap in the season-mapped
  version from `xWAR_hitters.ipynb`.
- To re-run this notebook for a different chunk of new data later, just change `HITTER_TEST_FILE` /
  `PITCHER_TEST_FILE` in the config cell and re-run — no other edits needed.